<a href="https://colab.research.google.com/github/zaheer-se/Accept_the_Technology/blob/main/Assignment_2_Questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# First import some necessary libararies
import numpy as np
import pandas as pd

The data is a corpus of 50K movie reviews (each as a document) saved in a list, and corresponding sentiment labels (0 = negative, 1 = postive) for each review saved in a 1D numpy array.   

- The data are saved in two pickle files, let's load them first with the following codes.

- For more details about the movie review data, please check [here](https://ai.stanford.edu/~amaas/data/sentiment/). Note that we have modified the data slightly for this assignment.

<font color=red>**In case you need to modify the data path to read them into the notebook properly, please copy the codes into another block and make modifications accordingly as the below codes are NOT editble.**</font>  

In [13]:
import pickle

with open("review_texts.pkl", 'rb') as f1:
    review_texts = pickle.load(f1)             # a list of 5000 documents

with open('review_labels.pkl', 'rb') as f2:
    review_labels = pickle.load(f2)            # 1D numpy.array (50000,)

# check the data shape and top 3 review/labels
display(len(review_texts), review_labels.shape, review_texts[:3], review_labels[:3])

50000

(50000,)

["Zero Day leads you to think, even re-think why two boys/young men would do what they did - commit mutual suicide via slaughtering their classmates. It captures what must be beyond a bizarre mode of being for two humans who have decided to withdraw from common civility in order to define their own/mutual world via coupled destruction.  It is not a perfect movie but given what money/time the filmmaker and actors had - it is a remarkable product. In terms of explaining the motives and actions of the two young suicide/murderers it is better than 'Elephant' - in terms of being a film that gets under our 'rationalistic' skin it is a far, far better film than almost anything you are likely to see.   Flawed but honest with a terrible honesty.",
 'Words can\'t describe how bad this movie is. I can\'t explain it by writing only. You have too see it for yourself to get at grip of how horrible a movie really can be. Not that I recommend you to do that. There are so many clichés, mistakes (and al

array([1, 0, 1])

We select the first 40K reviews and labels as training set, and the last 10K as the test set.

In [14]:
train_texts = review_texts[:40000]
test_texts = review_texts[40000:]

train_labels = review_labels[:40000]
test_labels = review_labels[40000:]

Please use the train and test data from the above blocks to complete the following tasks.   We aim to

- (1) explore two topic modeling algorithms (i.e., LSA and LDA) with the dataset ``train_texts``, and

- (2) vectorize the text data into sparse matrices using either `TF` or `TF-IDF`, subjective to the requirement of the topic model; and

- (3) predict the sentiment in each movie review, using either ``MultinomialNB`` model with `TF` or `TF-IDF` as features;  or ``GaussianNB`` model with latent topics as features.  

## Question 1: Latent Semantic Analysis and Sentiment Prediction


### 1.1 Text Vectorization (10 points)

Please train a vectorizer to transform the **train_texts** into a sparse matrix of tf-idf weights.  Please use `TfidfVectorizer` function from `scikit-learn` and set the parameters as the following:

- ``max_features`` = 10000
- ``lowercase`` = True
- ``ngram_range`` = (1,1)
- ``min_df`` = 2
- ``max_df`` = 0.5
- ``stop_words``='english'

Please name the vectorizer as **tfidf_vec** and the sparse matrix as **tfidf_train_docs**.  

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Initialize the TfidfVectorizer with the required parameters
tfidf_vec = TfidfVectorizer(
    max_features=10000,
    lowercase=True,
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.5,
    stop_words='english'
)

# 2. Train the vectorizer and transform the train_texts into a sparse matrix
# Assuming train_texts is the list of reviews from your previous block
tfidf_train_docs = tfidf_vec.fit_transform(train_texts)

# 3. Verification
print(f"Vectorizer trained with {len(tfidf_vec.get_feature_names_out())} features.")
print(f"Sparse matrix shape: {tfidf_train_docs.shape}")

Vectorizer trained with 10000 features.
Sparse matrix shape: (40000, 10000)


### 1.2 Sentiment Prediction with TF-IDF as Features (10 points)

We'd like to predict the sentiment in each movie review with the tf-idf features (i.e., **tfidf_train_docs**).  As the data follows exactly multinomial distribution,  we will use the Multinomial Naive Bayes classifier here.  

- Please use the ``MultinomialNB`` function from ``sklearn.naive_bayes`` module.  Check [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB) for details.
- Use all the default setting for the model, and name the model as **nb1**.
- Evaluate both the train and test performance of **nb1**.  

Note: In case the model require a dense matrix, convert the sparse matrix (e.g., ``M``) into a dense one with ``M.toarray()``.


In [16]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# 1. Initialize the Multinomial Naive Bayes model with default settings
nb1 = MultinomialNB()

# 2. Fit the model using the TF-IDF training features and labels
# Note: MultinomialNB can handle sparse matrices directly, so .toarray() is not required
nb1.fit(tfidf_train_docs, train_labels)

# 3. Predict and evaluate performance on the Training Set
train_preds = nb1.predict(tfidf_train_docs)
train_accuracy = accuracy_score(train_labels, train_preds)

# 4. Transform the Test Set and evaluate performance
# Important: Use .transform(), NOT .fit_transform() on the test data
tfidf_test_docs = tfidf_vec.transform(test_texts)
test_preds = nb1.predict(tfidf_test_docs)
test_accuracy = accuracy_score(test_labels, test_preds)

# Print the results
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Testing Accuracy: {test_accuracy:.2%}")

Training Accuracy: 87.45%
Testing Accuracy: 84.76%


### 1.3 Train a LSA Model (10 points)

Train a Latent Semantic Analysis model using the vectorized texts (i.e., **tfidf_train_docs**) to extract **10 latent topics**.

- Set the ``random_state`` as `0` for the ``TruncatedSVD`` algorithm. Name the model as **lsa**.
- Save the reduced doc-topic matrix as **doc_topic_lsa**, and the topic-term matrix (i..e, the topic model) as **topic_term_lsa**, check the shape of the two matrices respectively.

In [17]:
from sklearn.decomposition import TruncatedSVD

# 1. Initialize the TruncatedSVD model for LSA
# We set n_components=10 to extract 10 latent topics
lsa = TruncatedSVD(n_components=10, random_state=0)

# 2. Fit the model and transform tfidf_train_docs to get the Doc-Topic matrix
# This represents each document as a mixture of the 10 topics
doc_topic_lsa = lsa.fit_transform(tfidf_train_docs)

# 3. Access the Topic-Term matrix (the components)
# This represents the importance of each word (term) for each of the 10 topics
topic_term_lsa = lsa.components_

# 4. Check the shapes of the two matrices
print(f"Doc-Topic Matrix Shape (Documents x Topics): {doc_topic_lsa.shape}")
print(f"Topic-Term Matrix Shape (Topics x Terms): {topic_term_lsa.shape}")

Doc-Topic Matrix Shape (Documents x Topics): (40000, 10)
Topic-Term Matrix Shape (Topics x Terms): (10, 10000)


### 1.4 Interpret the Topics (10 points)

Select one topic as you like and describe its semantic meaning with less than 20 words and explain why briefly.

- You may use the ``top10_terms`` and ``top_doc`` functions defined in class codes, to obtain the top associated terms or document for each topic.

- There is NO standard answer regarding the interpretation.

In [18]:
def top10_terms(model, vectorizer):
    feature_names = vectorizer.get_feature_names_out()
    topic_term_matrix = model.components_
    n_topics = topic_term_matrix.shape[0]

    for topic_ind in np.arange(n_topics):
        terms = topic_term_matrix[topic_ind,:]
        term_ind = terms.argsort()[::-1]
        top_terms = feature_names[term_ind[:10]]
        print('Topic {}: {}'.format(topic_ind,top_terms))


def top_doc(model, vectorizer, raw_doc):
    vec_doc = vectorizer.transform(raw_doc)
    doc_topic_matrix = model.transform(vec_doc)
    no_topics = doc_topic_matrix.shape[1]

    for topic_ind in np.arange(no_topics):
        docs = doc_topic_matrix[:,topic_ind]
        doc_ind = docs.argsort()[::-1]
        print('Topic {}, highest weight on doc {} : '.format(topic_ind, doc_ind[0]))

        top_doc = raw_doc[doc_ind[0]]
        print(top_doc[:500].replace("\n", ""))
        print(' ')

In [19]:
top10_terms(model = lsa, vectorizer = tfidf_vec)

Topic 0: ['like' 'just' 'good' 'really' 'bad' 'time' 'story' 'don' 'great' 'movies']
Topic 1: ['bad' 'worst' 'movies' 'just' 'don' 'acting' 'really' 'watch' 'waste'
 'terrible']
Topic 2: ['great' 'love' 'funny' 'watch' 'think' 'really' 'loved' 'movies' 'series'
 'don']
Topic 3: ['good' 'great' 'story' 'acting' 'bad' 'actors' 'best' 'action' 'cast'
 'movies']
Topic 4: ['funny' 'good' 'really' 'comedy' 'pretty' 'guy' 'horror' 'fun' 'like'
 'girl']
Topic 5: ['story' 'really' 'book' 'good' 'life' 'people' 'bad' 'character' 'love'
 'characters']
Topic 6: ['series' 'episode' 'bad' 'book' 'tv' 'characters' 'episodes' 'funny'
 'character' 'season']
Topic 7: ['bad' 'funny' 'comedy' 'love' 'life' 'worst' 'man' 'movies' 'seen'
 'family']
Topic 8: ['bad' 'good' 'family' 'kids' 'old' 'love' 'series' 'children' 'horror'
 'story']
Topic 9: ['book' 'horror' 'just' 'read' 've' 'time' 'little' 'seen' 'family'
 'version']


In [20]:
top_doc(model = lsa, vectorizer = tfidf_vec, raw_doc = train_texts)

Topic 0, highest weight on doc 8187 : 
Once in a while, you come upon a movie that defines your values and shows you the true depth of human emotions leaving you drained. Vivah is just that  maybe more. After watching DDLJ, Saajan, and Lamhey, I really thought that Bollywood has reached its pinnacle and will never come up with anything like that - EVER. Boy was I wrong! I went to the store to buy some groceries and decided to pick this movie up along with the new "DON" (just so I can compare The Great AB with ShahRukh  although the
 
Topic 1, highest weight on doc 36051 : 
Crazy Six is torture, it must be Albert Pyun´s worst film. Even Blast and Ticker are better! I can´t believe how boring this film is! How this even got greenlighted? I saw this movie about 3 years ago and the only thing I remember is how bad it was. This isn´t good bad movie, it is simply bad, bad, bad, bad, bad movie.  1 out of 10 (½ out of *****)
 
Topic 2, highest weight on doc 38690 : 
this is the best movie i 

### 1.5 Sentiment Prediction with LSA Latent Topics as Features (10 points)

Let's use the topic features (i.e., **doc_topic_lsa**) to predict the sentiment of each movie review again.

With each document represented in the reduced space of latent topics, we can use all kinds of classification models learnt earlier.

- Here please use the ``GaussianNB`` function from `sklearn.naive_bayes` module. Gaussian Naive Bayes is suitable for continuous features. Check [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html) for details.

- Use all default setting for the ``GaussianNB``, name the model as **nb2**.
- Check the train and test accuracy of **nb2**.  

## Question 2: Latent Dirichlet Allocation  & Sentiment Prediction


In [21]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# 1. Initialize the Gaussian Naive Bayes model
nb2 = GaussianNB()

# 2. Fit the model using the LSA latent topics (doc_topic_lsa) and training labels
nb2.fit(doc_topic_lsa, train_labels)

# 3. Predict and evaluate performance on the Training Set
train_preds_lsa = nb2.predict(doc_topic_lsa)
train_accuracy_lsa = accuracy_score(train_labels, train_preds_lsa)

# 4. Transform the Test Set into the LSA topic space and evaluate
# Step A: Vectorize test text using the existing tfidf_vec
tfidf_test_docs = tfidf_vec.transform(test_texts)
# Step B: Transform the test docs into the 10-topic space using the existing lsa model
doc_topic_lsa_test = lsa.transform(tfidf_test_docs)

# Step C: Predict and calculate test accuracy
test_preds_lsa = nb2.predict(doc_topic_lsa_test)
test_accuracy_lsa = accuracy_score(test_labels, test_preds_lsa)

# Print the results
print(f"LSA-based Training Accuracy: {train_accuracy_lsa:.2%}")
print(f"LSA-based Testing Accuracy: {test_accuracy_lsa:.2%}")

LSA-based Training Accuracy: 78.43%
LSA-based Testing Accuracy: 78.55%



### 2.1 Text Vectorization (10 points)

Note that the LDA model expect term frequency input only.  

Please train a vectorizer to transform the **train_texts** into a sparse matrix of term frequency.  Please use `CountVectorizer` function from `scikit-learn` and set the parameters as the following:

- ``max_features`` = 10000
- ``lowercase`` = True
- ``ngram_range`` = (1,1)
- ``min_df`` = 2
- ``max_df`` = 0.5
- ``stop_words``= 'english'

Please name the vectorizer as **tf_vec** and the sparse matrix as **tf_train_docs**.



In [22]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Initialize the CountVectorizer with the required parameters
tf_vec = CountVectorizer(
    max_features=10000,
    lowercase=True,
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.5,
    stop_words='english'
)

# 2. Train the vectorizer and transform the train_texts into a sparse matrix
# This creates a matrix of raw word counts
tf_train_docs = tf_vec.fit_transform(train_texts)

# 3. Verification
print(f"Vectorizer trained with {len(tf_vec.get_feature_names_out())} features.")
print(f"Sparse matrix shape: {tf_train_docs.shape}")

Vectorizer trained with 10000 features.
Sparse matrix shape: (40000, 10000)


### 2.2 Sentiment Prediction with Term Frequency as Features (10 points)

With each movie review transformed into a sparse vector of term frequency, let's use the term frequency matrix (i.e., **tf_train_docs**) as features to predict the sentiment in each review.

- Again, please use ``MultinomialNB`` from ``sklearn.naive_bayes`` as the model, with all default setting.  Name the model as **nb3**.
- Check the model accuracy of **nb3** on both train and test data.  

Note: In case the model require a dense matrix, convert the sparse matrix (e.g., ``M``) into a dense one using ``M.toarray()``.

In [23]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# 1. Initialize the Multinomial Naive Bayes model
nb3 = MultinomialNB()

# 2. Fit the model using the Term Frequency training features (tf_train_docs)
nb3.fit(tf_train_docs, train_labels)

# 3. Predict and evaluate performance on the Training Set
train_preds_tf = nb3.predict(tf_train_docs)
train_accuracy_tf = accuracy_score(train_labels, train_preds_tf)

# 4. Transform the Test Set and evaluate performance
# Remember to use the tf_vec (CountVectorizer) created in 2.1
tf_test_docs = tf_vec.transform(test_texts)
test_preds_tf = nb3.predict(tf_test_docs)
test_accuracy_tf = accuracy_score(test_labels, test_preds_tf)

# Print the results
print(f"TF-based Training Accuracy: {train_accuracy_tf:.2%}")
print(f"TF-based Testing Accuracy: {test_accuracy_tf:.2%}")

TF-based Training Accuracy: 86.51%
TF-based Testing Accuracy: 84.22%


### 2.3  Train a LDA Model (10 points)

Please train another topic model using ``LatentDirichletAllocation`` function on the vectorized training text returned earlier (i.e., **tf_train_docs**), we expect **10 latent topics** in return.  

- Please name the model as **lda**, set ``random_state`` = 0.
- Save the reduced doc-topic matrix as **doc_topic_lda**, and the topic-term matrix (i..e, the topic model) as **topic_term_lda**, check the shape of two matrix respectively.

In [24]:
from sklearn.decomposition import LatentDirichletAllocation

# 1. Initialize the LDA model
# n_components=10 extracts 10 latent topics
lda = LatentDirichletAllocation(n_components=10, random_state=0)

# 2. Fit the model and transform the Term Frequency matrix
# This creates the Document-Topic matrix
doc_topic_lda = lda.fit_transform(tf_train_docs)

# 3. Access the Topic-Term matrix
# 'components_' stores the pseudocounts of words assigned to topics
topic_term_lda = lda.components_

# 4. Check the shapes of the two matrices
print(f"Doc-Topic Matrix Shape (Documents x Topics): {doc_topic_lda.shape}")
print(f"Topic-Term Matrix Shape (Topics x Terms): {topic_term_lda.shape}")

Doc-Topic Matrix Shape (Documents x Topics): (40000, 10)
Topic-Term Matrix Shape (Topics x Terms): (10, 10000)


### 2.4 Interpret the Topics (10 points)

Again, select one topic as you like and describe its semantic meaning with less than 20 words and explain why briefly.

- You may use the ``top10_terms`` and ``top_doc`` functions defined in class codes, to obtain the top associated terms or document for each topic.

- There is NO standard answer regarding the interpretation.

In [25]:
# 1. See the top words for the 10 LDA topics
top10_terms(model = lda, vectorizer = tf_vec)

# 2. See the most representative review for the 10 LDA topics
top_doc(model = lda, vectorizer = tf_vec, raw_doc = train_texts)

Topic 0: ['action' 'films' 'like' 'time' 'man' 'good' 'fight' 'story' 'best'
 'great']
Topic 1: ['people' 'war' 'life' 'world' 'story' 'time' 'family' 'years'
 'documentary' 'like']
Topic 2: ['series' 'episode' 'time' 'just' 'tv' 'like' 'episodes' 'bad' 'season'
 'worst']
Topic 3: ['great' 'best' 'role' 'performance' 'man' 'cast' 'john' 'love' 'story'
 'time']
Topic 4: ['like' 'just' 'bad' 'girl' 'really' 'good' 'old' 'sex' 'guy' 'scene']
Topic 5: ['good' 'just' 'like' 'really' 'movies' 'bad' 'time' 'watch' 'great' 'don']
Topic 6: ['like' 'just' 'people' 'don' 'know' 'think' 'time' 'make' 'really' 'did']
Topic 7: ['story' 'characters' 'character' 'life' 'love' 'like' 'films' 'way'
 'director' 'time']
Topic 8: ['good' 'plot' 'character' 'action' 'like' 'just' 'bad' 'does' 'police'
 'scene']
Topic 9: ['horror' 'like' 'just' 'bad' 'films' 'effects' 'really' 'make' 'budget'
 'good']
Topic 0, highest weight on doc 31272 : 
The Hand of Death aka Countdown in Kung Fu (1976) is a vastly underr

In [26]:
top_doc(model = lda, vectorizer = tf_vec, raw_doc = train_texts)

Topic 0, highest weight on doc 31272 : 
The Hand of Death aka Countdown in Kung Fu (1976) is a vastly underrated early work by director John Woo. The film stars Dorian Tan (Tan Tao-liang) and features Jackie Chan, Sammo Hung and James Tien in significant supporting roles. Many people believe, or have been lead to believe by deceptive advertising, that this is a Jackie Chan film. This is not a Jackie Chan film, Dorian Tan is the star but Jackie gives one of his best (most serious) early performances.  The Hand of Death is about a Shaol
 
Topic 1, highest weight on doc 3819 : 
I have been using IMDb for years and I never wanted to get involved in the commentary of moviesuntil now. This documentary has so many problems that I hardly know what to say. I am not a Muslim, nor am I an Islamic studies expert, but I know enough to shed some light on the obvious one-sided viewpoint that this documentary espouses.   The problems with this movie begin with the fact that it is a documentary. Most 

### 2.5 Sentiment Prediction with LDA Latent Topics as Features (10 points)

With latent topics extracted by the ``lda``  model, let's use them as features to predict the sentiment of each movie review.  

- Again, use  ``GaussianNB`` model with all default setting,  name the model as **nb4**.
- Check the train and test accuracy of **nb4**.  

In [27]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# 1. Initialize the Gaussian Naive Bayes model
nb4 = GaussianNB()

# 2. Fit the model using the LDA latent topics (doc_topic_lda) and training labels
nb4.fit(doc_topic_lda, train_labels)

# 3. Predict and evaluate performance on the Training Set
train_preds_lda = nb4.predict(doc_topic_lda)
train_accuracy_lda = accuracy_score(train_labels, train_preds_lda)

# 4. Prepare the Test Set features
# Step A: Vectorize test text using the existing tf_vec (CountVectorizer)
tf_test_docs = tf_vec.transform(test_texts)
# Step B: Transform the test docs into the 10-topic space using the existing lda model
doc_topic_lda_test = lda.transform(tf_test_docs)

# 5. Predict and evaluate performance on the Test Set
test_preds_lda = nb4.predict(doc_topic_lda_test)
test_accuracy_lda = accuracy_score(test_labels, test_preds_lda)

# Print the results
print(f"LDA-based Training Accuracy: {train_accuracy_lda:.2%}")
print(f"LDA-based Testing Accuracy: {test_accuracy_lda:.2%}")

LDA-based Training Accuracy: 68.72%
LDA-based Testing Accuracy: 68.15%


Congratulations! You have worked through the journey of topic modeling and used unsupervised model to improve supervised model. You may have noticed that both the vectorizer and the topic modeling algorithms are just the pre-processing steps for a predictive model. To explore the best value for one or multiple parameters (e.g., ``n_components`` in the LDA/LSA or ``max_df`` in the vectorizer), we can use `GridSearchCV` with both the vectorizer and the topic model algorithms as pre-processing steps for a supervised model, then find the best paramters based on the generalization performance of the last estimator.  In this process, we may need to encapsulate all pre-processing steps and the final model in a pipeline using ``Pipeline`` from ``sklearn.pipeline``.